# Aula 02: Heurísticas de Busca Local (Local Search)

Na aula anterior, aprendemos que algoritmos gulosos são rápidos, mas frequentemente ficam presos em soluções sub-ótimas. Nesta aula, exploraremos a **Busca Local**, uma técnica que não constrói a solução do zero, mas tenta melhorar uma solução já existente.

## 1. O Conceito Fundamental e o Espaço de Busca

O primeiro passo é definir o vocabulário básico para operarmos no "escuro" do espaço de busca:

- **Solução Inicial ($S_0$)**: É o nosso ponto de partida. Pode ser uma solução gerada aleatoriamente ou, de forma mais inteligente, o resultado de uma heurística gulosa (como o Vizinho Mais Próximo).
- **Espaço de Busca**: O conjunto de todas as soluções possíveis para o problema. Para problemas reais, este espaço é tão vasto que é impossível mapeá-lo inteiramente.
- **Função de Avaliação ($f(S)$)**: O nosso "termômetro" ou *fitness*. É o cálculo que nos diz o quão boa é a solução atual (ex: a distância total da rota no TSP ou o lucro total na Mochila).

## 2. A Estrutura de Vizinhança ($N(s)$)

Este é o conceito mais importante da aula. Para o computador, proximidade não é distância física, mas sim a **quantidade de alterações** necessárias para transformar uma solução em outra.

### Exemplos de Vizinhanças:
1. **Mochila (Binário)**: Vizinhança *Bit-flip* (inverter a decisão de um único item de 0 para 1, ou vice-versa).
2. **TSP (Permutação)**:
   - **Swap**: Trocar a posição de duas cidades na rota.
   - **2-opt**: Quebrar duas arestas e reconectá-las de forma cruzada. É o exemplo visual perfeito para desfazer cruzamentos desnecessários.

## 3. Estratégias de Movimento (Pivoteamento)

Quando olhamos para os "vizinhos", como escolhemos para qual deles ir?

- **Primeira Melhora (First Improvement)**: O algoritmo testa os vizinhos um a um. Assim que encontra o primeiro vizinho melhor que a solução atual, ele se move imediatamente. É muito rápido computacionalmente.
- **Melhor Melhora (Best Improvement)**: O algoritmo testa **todos** os vizinhos possíveis, compara-os e escolhe aquele que oferece o maior ganho absoluto. Exige mais processamento, mas tende a dar saltos maiores de qualidade por passo.

## 4. Implementando a Busca Local 2-opt no TSP

Vamos aplicar a estratégia de **Melhor Melhora** usando a vizinhança 2-opt.

In [ ]:
import math
import matplotlib.pyplot as plt

def dist(c1, c2):
    return math.sqrt((c1['x'] - c2['x'])**2 + (c1['y'] - c2['y'])**2)

def calc_total_dist(tour):
    d = sum(dist(tour[i], tour[i+1]) for i in range(len(tour)-1))
    d += dist(tour[-1], tour[0])
    return d

def two_opt_swap(tour, i, k):
    return tour[:i] + tour[i:k+1][::-1] + tour[k+1:]

def local_search_2opt(tour):
    best_tour = tour
    best_dist = calc_total_dist(tour)
    improved = True
    
    while improved:
        improved = False
        # Explorando a vizinhança completa (Best Improvement)
        for i in range(1, len(best_tour) - 1):
            for k in range(i + 1, len(best_tour)):
                new_tour = two_opt_swap(best_tour, i, k)
                new_dist = calc_total_dist(new_tour)
                
                if new_dist < best_dist:
                    best_dist = new_dist
                    best_tour = new_tour
                    improved = True
        
    return best_tour, best_dist

# Exemplo: Rota com cruzamento (0 -> 1 -> 2 -> 3 -> 0)
cidades = [{'id':0, 'x':0, 'y':0}, {'id':1, 'x':10, 'y':10}, {'id':2, 'x':0, 'y':10}, {'id':3, 'x':10, 'y':0}]
rota_otimizada, dist_final = local_search_2opt(cidades)

print(f"Distância Final Pós Otimização: {dist_final:.2f}")

## 5. O Ótimo Local

 O que acontece quando o algoritmo olha para todos os vizinhos e **nenhum de deles é melhor** que a solução atual?

<div align="center">
  <img src="img/ls.png" width="500px" />
  <p><i>Visualização de Ótimos Locais vs Ótimo Global em uma superfície de custo.</i></p>
</div>

O algoritmo para. Ele atingiu um **Ótimo Local**.

### A Analogia do Alpinista (Hill Climbing)
Imagine um alpinista tateando o chão em um dia de neblina espessa. Ele sempre dá um passo para onde o terreno sobe. Quando sente que todos os lados ao seu redor descem, ele conclui que chegou ao topo do Everest (**Ótimo Global**). No entanto, ele pode estar apenas no topo de um pequeno morro vizinho.

**Outras armadilhas:**
- **Platôs**: Áreas planas onde todos os vizinhos têm o mesmo valor, e a busca anda em círculos.
- **Vales Estreitos**: Onde o progresso é extremamente lento.

## 6. Escapando das Armadilhas 

Como consertar o defeito  da Busca Local Clássica? Introduzimos estratégias para "saltar" para fora dos ótimos locais:

1. **VNS (Variable Neighborhood Search)**: Se você travou em um ótimo local usando 2-opt, tente uma vizinhança maior (ex: 3-opt, trocando 3 cidades) para tentar cair em outra "montanha" do espaço de busca.
2. **ILS (Iterated Local Search)**: Aplique um "chute" (perturbação) na solução atual. Destrua uma parte da rota aleatoriamente e mande a busca local tentar consertar.
3. **Busca Tabu**: Mantenha uma lista de movimentos proibidos recentemente (memória de curto prazo) para forçar o algoritmo a explorar áreas novas e evitar que ele fique indo e voltando entre os mesmos vizinhos.

Nas próximas aulas, implementaremos a metaheurística **Simulated Annealing**, que aceita soluções piores temporariamente para escapar desses vales!